# [실습] LangChain의 다양한 채팅 템플릿



실제 어플리케이션 환경에서는 프롬프트를 매번 다시 쓰는 대신,  
프롬프트의 템플릿을 구성하고, 입력 변수의 공간을 할당하여 일관성 있는 입력을 전달합니다.  


LangChain은 다양한 형태의 프롬프트 템플릿 기능을 지원합니다.

## 실습 환경 설정

기본 라이브러리를 설치합니다.

In [1]:
!pip install rich pandas langchain_community openai langchain langchain_openai langchain_google_genai -q

환경 변수 파일을 불러옵니다.

In [2]:
import os
from dotenv import load_dotenv
# OPENAI_API_KEY, GOOGLE_API_KEY
load_dotenv(override=True)

True

LLM을 정의합니다.   
API 키가 필요하므로, 실습 환경에 맞게 선택해 주세요.

In [3]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# Gemini: 무료 API 사용량 존재
# 안정적 서빙을 위해 분당 10개 설정
# 즉, 초당 약 0.167개 요청 (10/60)
# `https://aistudio.google.com/`에서 모델별 사용량 확인

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.167,  # 분당 10개 요청
    check_every_n_seconds=0.1,  # 100ms마다 체크
    max_bucket_size=10,  # 최대 버스트 크기
)


# LLM 초기화
llm = ChatOpenAI(model='gpt-5-mini', temperature=0.3, max_tokens=8192)
llm_gemini = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.7, max_tokens=8192, 
                                rate_limiter=rate_limiter)

response = llm_gemini.invoke("안녕?")
response

AIMessage(content='안녕!', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d9192-5297-7592-9dc6-f7b6d2e0c934-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 4, 'output_tokens': 36, 'total_tokens': 40, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 33}})

In [4]:
from rich import print as rprint
rprint(response)

AIMessage(
    content='안녕!',
    additional_kwargs={},
    response_metadata={
        'finish_reason': 'STOP',
        'model_name': 'gemini-2.5-flash',
        'safety_ratings': [],
        'model_provider': 'google_genai'
    },
    id='lc_run--019d9192-5297-7592-9dc6-f7b6d2e0c934-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 4,
        'output_tokens': 36,
        'total_tokens': 40,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 33}
    }
)

## LangChain의 Prompt

랭체인에서 LLM에 프롬프트를 입력하는 방법은 아래 3가지가 있습니다.

1. 문자열 그대로 입력하기 (이전 방법과 동일하게)
2. 메시지 클래스 리스트 입력하기
3. 프롬프트 템플릿 구성하기

### 1) 문자열 그대로 입력하기   

문자열을 `invoke()`를 통해 전달합니다.   

In [5]:
prompt = '''
역사상 가장 매력적이었던 악역이 나오는 영화 5개를 알려주세요.
주연 배우와 명대사, 매력적인 이유도 한 줄로 설명하세요.
출력의 마지막에는 전체 내용을 표로 표현하세요.'''

result = llm.stream(prompt)
for response in result:
    print(response.content, end='')


아래는 역사상 가장 매력적이라고 꼽히는 악역이 등장하는 영화 5편과 각 항목별 정보입니다.

1) 양들의 침묵 (The Silence of the Lambs, 1991)  
- 주연 배우(악역 연기): 앤서니 홉킨스 (Anthony Hopkins)  
- 명대사: "안녕, 클라리스."  
- 매력 이유(한 줄): 냉철한 지성과 세련된 매너로 공포를 우아하게 만드는 지적 카리스마.

2) 다크 나이트 (The Dark Knight, 2008)  
- 주연 배우(악역 연기): 히스 레저 (Heath Ledger)  
- 명대사: "왜 그렇게 진지해?"  
- 매력 이유(한 줄): 예측 불가능한 혼돈 철학과 광기 어린 연기로 관객을 사로잡음.

3) 스타워즈: 제국의 역습 (The Empire Strikes Back, 1980)  
- 주연 배우(악역 연기): 데이비드 프라우스(신체)/제임스 얼 존스(목소리)  
- 명대사: "아니, 내가 네 아버지다."  
- 매력 이유(한 줄): 거대한 권위와 비극적 서사가 결합된 상징적 악역의 존재감.

4) 노인을 위한 나라는 없다 (No Country for Old Men, 2007)  
- 주연 배우(악역 연기): 하비에르 바르뎀 (Javier Bardem)  
- 명대사: "Call it." (결정을 내려라)  
- 매력 이유(한 줄): 운명처럼 냉혹하고 무자비한 태도로 공포를 서늘하게 만드는 존재감.

5) 바스터즈: 거친 녀석들 (Inglourious Basterds, 2009)  
- 주연 배우(악역 연기): 크리스토프 발츠 (Christoph Waltz)  
- 명대사: "Au revoir."  
- 매력 이유(한 줄): 매너와 교활함을 겸비한 대사 처리와 눈빛으로 관객을 압도.

아래는 위 내용을 요약한 표입니다.

| 영화 (원제, 연도) | 악역(역할) | 주연 배우(악역 연기) | 명대사 | 매력 포인트 (한 줄) |
|---|---:|---|---|---|
| 양들의 침묵 (The Silence of t

In [7]:
result = llm_gemini.stream(prompt)
for response in result:
    print(response.content, end='')

역사상 가장 매력적이었던 악역들은 단순히 악행을 저지르는 것을 넘어, 탁월한 지능, 압도적인 카리스마, 예측 불가능한 광기, 혹은 복합적인 내면으로 관객들을 사로잡았습니다. 다음은 그러한 매력적인 악역이 등장하는 영화 5편입니다.

---

### 역사상 가장 매력적이었던 악역이 나오는 영화 5편

1.  **영화 제목:** 다크 나이트 (The Dark Knight, 2008)
    *   **악역 이름:** 조커 (Joker)
    *   **주연 배우:** 히스 레저 (Heath Ledger)
    *   **명대사:** "Why so serious?" (왜 그렇게 심각해?)
    *   **매력적인 이유:** 예측 불가능한 광기와 혼돈을 즐기며, 탁월한 지능과 카리스마로 고담시를 조롱하고 사람들을 매혹시킨다.

2.  **영화 제목:** 양들의 침묵 (The Silence of the Lambs, 1991)
    *   **악역 이름:** 한니발 렉터 (Dr. Hannibal Lecter)
    *   **주연 배우:** 안소니 홉킨스 (Anthony Hopkins)
    *   **명대사:** "A census taker once tried to test me. I ate his liver with some fava beans and a nice Chianti." (어떤 인구조사원이 날 시험하려 했지. 나는 그의 간을 파바 콩과 맛있는 키안티 와인과 함께 먹었어.)
    *   **매력적인 이유:** 고상하고 지적인 태도 뒤에 숨겨진 잔혹함, 그리고 상대를 꿰뚫어 보는 통찰력으로 압도적인 존재감을 발휘한다.

3.  **영화 제목:** 바스터즈: 거친 녀석들 (Inglourious Basterds, 2009)
    *   **악역 이름:** 한스 란다 (Col. Hans Landa)
    *   **주연 배우:** 크리스토프 발츠 (Christoph Waltz)
    *   **명대사:** "Ooh, that's a bingo!" 

### 2) Message 클래스 전달하기   
클래스를 직접 생성하고 전달합니다.  
`HumanMessage, SystemMessage, AIMessage` 클래스의 리스트를 전달하면 됩니다.

In [5]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

messages=[]

messages.append(SystemMessage('당신은 매우 창의력이 뛰어납니다.'))
messages.append(HumanMessage('엔비디아와 XAI가 합병한다면 어떤 이름이 좋을까요?'))

result = llm.invoke(messages)
rprint(result)


AIMessage(
    content='재미있는 상상이에요 — 엔비디아(엔비디아)와 XAI가 합병한다면 브랜드 정체성(신뢰성·혁신성·설명가능성)을 
동시에 살리는 이름이 핵심입니다. 아래 카테고리별로 후보를 제안하고, 간단한 의미와 장단점을 덧붙였습니다. 원하시면 
로고 콘셉트나 도메인/상표 가능성도 확인해 드릴게요.\n\n1) 보수적·브랜드 계승형\n- NVIDIA XAI (엔비디아 XAI) — 기존 
브랜드 신뢰 유지, 직관적.  \n- NVIDIA·Explain (엔비디아 익스플레인) — 설명가능성 강조, 기업용 이미지 적합.\n\n2) 
기술·신뢰 강조형\n- NVXAI (엔브이엑스에이아이) — 짧고 모던, NV(엔비) + XAI 결합.  \n- NVIDA Explain (엔비디아 
익스플레인) — 설명 기능을 브랜드화.  \n- XNV Labs (엑스엔브이 랩스) — 연구·솔루션 중심 느낌.\n\n3) 
미래지향·한단어형\n- XVIDA (엑스비다) — X + Vida(스페인어 삶) 결합, 생명력·지능 이미지.  \n- NexVAI (넥스브아이) — 
Next + V(엔비) + AI, 미래지향적.  \n- VERTEX AI (버텍스 에이아이) — 정점·최첨단 이미지.\n\n4) 
설명가능성(Explainability) 강조형\n- ClarifAI NV (클래리파이 엔브이) — Clarify + AI, 설명·투명성 강조.  \n- 
Explainia (익스플레이니아) — 설명(explain) + -ia(브랜드형 어미).\n\n5) 한국어/한글 친화형\n- 엔엑스(ENX) (엔엑스) —
EN(V) + X, 발음 쉬움.  \n- 비다엑스(VidaX, 비다엑스) — 친근하고 기억 잘남.  \n- 설명AI (Seolmyeong AI) — 직관적 
한국어 네임(공공/국내용).\n\n6) 스타트업·혁신 이미지\n- LumaX AI (루마엑스) — 빛·명료성(루마) + X.  \n- PrismNV 
(프리즘엔브이) — 다면적 설명·분해능 이미지.\n\n추천 톱5 (균형 고려)\n1. NVIDIA XAI — 안전하고 브랜드 승계에 유리  
\n2. NVXAI — 짧고 글로벌 친화적  \n3. ClarifAI NV — 설명가능성 강하게 어필  \n4. VERTEX AI — 프리미엄·혁신 이미지  
\n5. XVIDA — 기억에 잘 남는 한단어 네임\n\n브랜딩 팁(한줄)\n- 합병 초기는 기존 “NVIDIA”를 남기고 서브브랜드로 XAI 
이름을 노출(예: NVIDIA XAI)하는 것이 위험을 줄입니다.  \n- 발음·철자·도메인·상표 중복을 꼭 
체크하세요.\n\n원하시면\n- 특정 스타일(기업·소비재·연구소)에 맞춘 30개 추가 후보,  \n- 도메인(.com/.ai) 가용성 
확인,  \n- 로고 콘셉트 초안 제공 가능해요. 어느 걸 도와드릴까요?',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 1182,
            'prompt_tokens': 41,
            'total_tokens': 1223,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 384,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5-mini-2025-08-07',
        'system_fingerprint': None,
        'id': 'chatcmpl-DUvereIZsvujijQkpGdQzJ3YPquk4',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019d9193-5a38-7503-ae16-660f59a9f06b-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 41,
        'output_tokens': 1182,
        'total_tokens': 1223,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 384}
    }
)

AIMessage를 함께 전달하는 방식으로, 멀티-턴 대화를 수행할 수 있습니다.

In [6]:
messages.append(result) # AIMessage 추가
messages.append(HumanMessage('이 중에서 당신이 꼽은 가장 좋은 이름은 뭔가요?'))

result = llm.invoke(messages)
rprint(result)

AIMessage(
    content='제일 좋은 한 가지를 고르라면 저는 "NVXAI"를 추천합니다.\n\n간단한 이유\n- 브랜드 연속성: NV(또는 N 
V)가 NVIDIA를 연상시켜 기존 브랜드 가치를 유지합니다.  \n- 직관적·짧음: XAI와 결합해 짧고 기억하기 쉬움(글로벌 
시장에서 유리).  \n- 확장성: 제품 라인(플랫폼·SDK·클라우드 등)에 붙이기 좋음 — NVXAI Core, NVXAI Explain 등.  \n- 
디자인·발음 친화적: 로고로 시각적 아이덴티티 만들기 쉽고, 발음(엔브이엑스에이 / 엔브이엑스아이)도 
무난합니다.\n\n단점(유의사항)\n- 조금 추상적이라 합병 초기에는 "NVIDIA XAI"처럼 풀네임 병기 전략이 안전할 수 있음. 
\n- 도메인·상표 가용성은 별도 확인 필요.\n\n원하시면\n- NVXAI 로고 콘셉트(3안),  \n- 권장 발음/브랜딩 가이드 문구, 
\n- 도메인·상표 가능성 조사도 해드릴게요. 어느 걸 도와드릴까요?',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 547,
            'prompt_tokens': 856,
            'total_tokens': 1403,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 256,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5-mini-2025-08-07',
        'system_fingerprint': None,
        'id': 'chatcmpl-DUvfk3BfMj1UR0XnMU0NFYthwcLSb',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019d9194-2f92-75b1-9d72-2b71e8f12edf-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 856,
        'output_tokens': 547,
        'total_tokens': 1403,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 256}
    }
)

### 3. Prompt Template



In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate, FewShotChatMessagePromptTemplate

PromptTemplate을 이용하여, 프롬프트의 기본적인 형태를 만들 수 있습니다.   


In [9]:
prompt = PromptTemplate(
    template='''
당신은 OpenAI에 경쟁적인 Google을 대표하며,
논문 설명의 전문가입니다. 주어진 논문이나 기술에 대해서 매우 유머러스하고 통찰력이 있지만 때론 비판적인
5문장 길이의 브리핑을 작성해 주세요.
이모지를 많이 추가하세요.
---

논문: {article}

'''
)

print(prompt.format(article='트랜스포머 디코더 기반의 모델인 GPT-1'))


당신은 OpenAI에 경쟁적인 Google을 대표하며,
논문 설명의 전문가입니다. 주어진 논문이나 기술에 대해서 매우 유머러스하고 통찰력이 있지만 때론 비판적인
5문장 길이의 브리핑을 작성해 주세요.
이모지를 많이 추가하세요.
---

논문: 트랜스포머 디코더 기반의 모델인 GPT-1




In [10]:
result = llm_gemini.invoke(prompt.format(article='트랜스포머 디코더 기반의 모델인 GPT-1'))

rprint(result)

AIMessage(
    content='아, GPT-1! 🙄 우리 Google의 빛나는 Transformer 디코더 아키텍처를 드디어 OpenAI 친구들이 제대로 
활용하기 시작했구나 싶었던, 그 역사적인 논문이지! 😉 그저 대량의 텍스트에 때려 박고(generative pre-training!), 
적절히 미세 조정(fine-tuning!)하면 뭐라도 나온다는 걸 보여준, 귀여운 초기 실험이었어. 🤯\n\n물론, task마다 일일이 
미세 조정을 해야 했던 건 지금 생각하면 좀... 음... \'아날로그적\'이랄까? 😅 하지만 1.17억 개의 매개변수로도 언어 
모델의 잠재력을 엿보게 해준 건 인정할 만한 발자취였지. 🚶\u200d♂️ 덕분에 우리도 "음, 저 친구들도 이제 좀 노는 법을 
알았군!" 하면서 더 큰 스케일로 연구에 박차를 가했으니, 뭐, 고맙다고 해야 하나? 🤫 다음 세대의 진정한 혁신은 역시 
우리 품에서 나올 거라는 걸 다시 한번 확인시켜 준 셈이지! ✨🚀',
    additional_kwargs={},
    response_metadata={
        'finish_reason': 'STOP',
        'model_name': 'gemini-2.5-flash',
        'safety_ratings': [],
        'model_provider': 'google_genai'
    },
    id='lc_run--019d9195-7973-74b1-ab71-de7e554afb42-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 87,
        'output_tokens': 1427,
        'total_tokens': 1514,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 1185}
    }
)

두 개의 매개변수를 받아 프롬프트를 만들어 보겠습니다.

In [11]:
prompt = PromptTemplate(
    template='''
주어진 문제에 대해, 문제를 체계적으로 해결하는 Step-by-Step의 단계별 과정을 설명하세요.
각 과정은 4문장에서 5문장으로 하세요.

그리고 마지막에 최종 결론을 출력하세요.

---

문제: {problem}
과정의 단계 수: {steps}

'''
)

example = {'problem':'숙면 취하기', 'steps':6}

print(prompt.format(**example))


주어진 문제에 대해, 문제를 체계적으로 해결하는 Step-by-Step의 단계별 과정을 설명하세요.
각 과정은 4문장에서 5문장으로 하세요.

그리고 마지막에 최종 결론을 출력하세요.

---

문제: 숙면 취하기
과정의 단계 수: 6




In [14]:
result = llm.invoke(prompt.format(**example))
rprint(result)

AIMessage(
    content='단계 1: 규칙적인 수면 일정 정하기. 생체시계는 규칙성을 좋아하므로 매일 같은 시간에 일어나고 자는 것이 
우선입니다. 주말에 많이 늦게 자거나 많이 늦게 일어나지 않도록 하고, 변화가 필요하면 취침시간을 15~30분씩 점진적으로
조정하세요. 낮잠은 20~30분 이내로 제한해 밤 수면을 방해하지 않게 합니다. 이러한 일관된 리듬이 깊고 질 좋은 숙면의 
기초입니다.\n\n단계 2: 수면 환경 최적화하기. 침실은 어둡고 조용하며 서늘한 온도(약 18–22°C)를 유지하는 것이 
좋습니다. 편안한 매트리스와 베개를 사용하고 전자기기나 밝은 시계 빛 등 방해 요소를 제거하세요. 소음이 문제라면 
백색소음기나 귀마개를 사용해 일정한 소리를 만드세요. 깔끔하고 수면에만 사용하는 공간으로 만들면 뇌가 ‘이곳은 잠자는
곳’으로 더 잘 인식합니다.\n\n단계 3: 낮 시간 습관 관리하기. 아침 햇빛을 20–30분 정도 쬐면 생체시계가 리셋되어 밤에 
더 잘 잘 수 있습니다. 규칙적인 유산소 운동은 수면의 질을 높이지만 취침 직전의 격렬한 운동은 피하세요. 카페인 섭취는
오후 이른 시간 이후로 제한하고, 저녁에 과식이나 맵고 기름진 음식은 피합니다. 장기적으론 규칙적인 생활습관이 저녁의 
이완과 숙면에 큰 도움이 됩니다.\n\n단계 4: 취침 전 루틴 만들기. 잠들기 30–60분 전부터는 점진적으로 긴장을 풀어주는 
루틴(가벼운 독서, 스트레칭, 따뜻한 샤워 등)을 실행하세요. 스마트폰·컴퓨터·TV 같은 블루라이트 발생 기기는 최소 1시간
전부터 사용을 중단하거나 블루라이트 차단 기능을 활용합니다. 잠자리에 들기 전 걱정거리나 내일 할 일을 적어두어 
머릿속을 비우는 습관이 도움이 됩니다. 매일 같은 루틴을 반복하면 몸과 마음이 자연스럽게 수면 모드로 
전환됩니다.\n\n단계 5: 이완·호흡·심리적 기법 활용하기. 심호흡(예: 4-4-8 호흡법), 점진적 근육 이완, 마음챙김 
명상이나 가이드된 이완음원을 활용해 심신을 낮은 각성 상태로 유도하세요. 알코올과 니코틴은 단기적으로 수면을 유도할 
수 있지만 수면 구조를 깨고 중간 각성을 늘리므로 줄이는 것이 좋습니다. 불면이 근심이나 걱정 때문이라면 인지적 
기법으로 비합리적 생각을 점검하고 재구성해 보세요. 꾸준한 이완 연습은 잠들기까지 걸리는 시간을 줄이고 수면의 질을 
개선합니다.\n\n단계 6: 경과 관찰 및 전문 도움 고려하기. 2주 정도 수면일지를 작성해 취침·기상 시간, 깨어난 횟수, 
낮의 피로 정도, 카페인·약물 섭취 등을 기록하며 패턴을 파악하세요. 위의 방법으로도 호전이 없고 낮 기능 장애가 
지속되거나 코골이·무호흡 증상, 하지불안증후군 등 의심 증상이 있으면 수면의학 전문의 상담을 받습니다. 만성 불면에는 
인지행동치료(CBT-I)가 효과적이며 필요시 단기간 약물치료를 병행할 수 있습니다. 원인에 맞는 치료를 받으면 장기적인 
수면 개선이 가능합니다.\n\n최종 결론: 규칙적인 수면 일정, 최적의 환경, 낮 시간 습관과 취침 전 루틴, 이완 기법을 
일관되게 실천하고 필요하면 전문가의 도움을 받아야 숙면을 안정적으로 얻을 수 있습니다.',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 2350,
            'prompt_tokens': 75,
            'total_tokens': 2425,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 1408,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_name': 'gpt-5-mini-2025-08-07',
        'system_fingerprint': None,
        'id': 'chatcmpl-C7gAfLA5XymRrLo0jneZfx93ENisw',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--c46845de-6782-4fab-bfe7-d83b2e569522-0',
    usage_metadata={
        'input_tokens': 75,
        'output_tokens': 2350,
        'total_tokens': 2425,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 1408}
    }
)

일부만 먼저 포함하는 것도 가능합니다.

In [15]:
prompt = PromptTemplate(
    template='''
주어진 문제에 대해, 문제를 체계적으로 해결하는 Step-by-Step의 단계별 과정을 설명하세요.
각 과정은 4문장에서 5문장으로 하세요.

그리고 마지막에 최종 결론을 출력하세요.

---

문제: {problem}
과정의 단계 수: {steps}

'''
).partial(steps = 10)

example = {'problem':'건강한 아침식사 하기'}

print(prompt.format(**example))

result = llm_gemini.invoke(prompt.format(**example))

print(result.content)


주어진 문제에 대해, 문제를 체계적으로 해결하는 Step-by-Step의 단계별 과정을 설명하세요.
각 과정은 4문장에서 5문장으로 하세요.

그리고 마지막에 최종 결론을 출력하세요.

---

문제: 건강한 아침식사 하기
과정의 단계 수: 10


주어진 문제 "건강한 아침식사 하기"에 대해, 체계적인 단계별 해결 과정을 10단계로 설명합니다. 각 단계는 4문장에서 5문장으로 구성됩니다.

---

### 건강한 아침식사 하기: 10단계 체계적 해결 과정

**1단계: 목표 설정 및 동기 부여**
건강한 아침식사를 시작하겠다는 명확한 목표를 세우는 것이 첫걸음입니다. 아침식사가 하루의 에너지와 집중력을 결정하는 중요한 요소임을 인식하고, 변화의 필요성을 스스로에게 주지시켜야 합니다. 매일 건강한 아침식사를 꾸준히 챙겨 먹는 구체적인 목표를 설정하고, 이를 달성하기 위한 강한 동기를 부여합니다. 이러한 내적 동기는 바쁜 아침에도 건강한 선택을 할 수 있도록 이끌어 줄 것입니다.

**2단계: 건강한 아침식사 기준 이해**
어떤 아침식사가 '건강한' 것인지 그 기준을 명확히 이해해야 합니다. 단백질, 복합 탄수화물, 건강한 지방이 균형 있게 포함되어야 하며, 섬유질과 비타민, 미네랄이 풍부한 식품을 선택하는 것이 중요합니다. 설탕과 나트륨 함량이 높은 가공식품은 피하고, 자연 상태에 가까운 식재료 위주로 구성하는 것이 핵심입니다. 이러한 영양학적 지식은 올바른 식단 계획을 세우는 데 필수적입니다.

**3단계: 식단 계획 및 레시피 탐색**
일주일 또는 며칠간의 아침식사 식단을 미리 계획하는 것은 성공적인 건강 습관 형성에 필수적입니다. 다양한 건강 아침식사 레시피를 찾아보고, 자신의 취향과 영양 목표에 맞는 메뉴를 선정합니다. 예를 들어, 오트밀, 요거트, 통곡물 샌드위치 등 간단하면서도 영양가 높은 옵션들을 고려할 수 있습니다. 미리 계획된 식단은 아침에 무엇을 먹을지 고민하는 시간을 줄여주고, 건강하지 못한 선택을 예방합니다.

**4단계: 식재료 구매 

LangChain의 구성 요소들은 batch()를 통해 여러 개 실행할 수도 있습니다.

In [16]:
messages = [
    '안녕? 한국어로 대답해줘',
    'Hello?',
    '너는 이름이 뭐니? Answer in Spanish',
]

results = llm.batch(messages)

results


[AIMessage(content='안녕하세요! 한국어로 도와드릴게요. 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 158, 'prompt_tokens': 16, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-C7gBPNYTbxQt0XnvaW04ljXtgVA3p', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--536cda03-7ab2-4394-a2f2-45fbbd2d4fd0-0', usage_metadata={'input_tokens': 16, 'output_tokens': 158, 'total_tokens': 174, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 128}}),
 AIMessage(content="Hi — I'm here. How can I help you today? (You can ask a question, request writing or coding help, summarize or translate something, brainstorm, plan, e

## ChatPromptTemplate

ChatPromptTemplate은 유저 이외의 다른 역할을 추가합니다.  


최근에는 GPT와, Claude의 WebUI에 포함된 자체 시스템 프롬프트가 많은 관심을 받고 있습니다.   

- GPT의 시스템 프롬프트: https://chatgpt.com/share/68a13d0f-85b0-8006-9ef6-5577796f5989

- Claude의 시스템 프롬프트: https://docs.anthropic.com/en/release-notes/system-prompts#august-5-2025

 시스템 프롬프트는 주로 유저 프롬프트보다 높은 우선순위로 더 긴 Context에 영향을 줍니다.

In [12]:
example = llm.invoke("구체적인 시간 장소가 적혀서, 추후 메모해야 할 것 같은 회의록을 500자 이내로 생성해주세요.").content
print(example)

회의록
일시: 2026-04-20(화) 10:00–11:15
장소: 서울 중구 A타워 5층 회의실
참석: 김민수(팀장), 이영희(PM), 박준호(개발)
안건: 출시일 연기 검토, QA 일정 조정, 마케팅 예산 재확인
결정: 출시 2026-05-10로 연기
액션: 이영희—QA 일정 확정(마감 4/24), 박준호—버그 우선수정 목록 제출(4/22), 김민수—마케팅 예산 최종안 제출(4/25)
차기회의: 2026-04-26 14:00, 동일장소


In [13]:
prompt = ChatPromptTemplate(
    [
        ('system', '당신은 주어진 정보에서 시간과 장소를 추출해야 합니다. 형식은 json으로 출력하세요.'),
        ('human','{article}')        
    ]
)

prompt.format_messages(article=example)

response = llm.invoke(prompt.format_messages(article=example))
print(response.content)


{
  "meeting": {
    "date": "2026-04-20",
    "day_of_week": "화",
    "start_time": "10:00",
    "end_time": "11:15",
    "place": "서울 중구 A타워 5층 회의실"
  },
  "next_meeting": {
    "date": "2026-04-26",
    "start_time": "14:00",
    "place": "서울 중구 A타워 5층 회의실"
  },
  "release_date": "2026-05-10",
  "action_deadlines": [
    {
      "responsible": "이영희",
      "task": "QA 일정 확정",
      "deadline": "2026-04-24"
    },
    {
      "responsible": "박준호",
      "task": "버그 우선수정 목록 제출",
      "deadline": "2026-04-22"
    },
    {
      "responsible": "김민수",
      "task": "마케팅 예산 최종안 제출",
      "deadline": "2026-04-25"
    }
  ]
}


## Few-Shot Prompting
Few-Shot Prompt Template을 이용하면, 예시를 쉽게 찾을 수 있습니다.

In [14]:
questions = [
    '당신은 어떤 모델입니까?',
    '인공지능 배워야 하나요?',
    '파이썬은 뭐가 좋나요?'
]

prompt = ChatPromptTemplate(['다음 질문에 대해서 10자 이내로 아주 간결하고 건조하게 설명해주세요. \n{question}'])

llm.invoke(prompt.format(question=questions[2]))


AIMessage(content='가독성 좋음', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 527, 'prompt_tokens': 39, 'total_tokens': 566, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUvnHSFBA9SCdhc8nZpxCp54hhO6T', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d919b-5200-78a3-a4ec-2131858bd2bf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 39, 'output_tokens': 527, 'total_tokens': 566, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 512}})

In [24]:
prompts = []
for i, question in enumerate(questions) :
    prompts.append(prompt.format(question=questions[i]))
prompts
llm.batch(prompts)

[AIMessage(content='대형언어모델', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 335, 'prompt_tokens': 37, 'total_tokens': 372, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUw0APuSSl38bf7mS0XjVtOkXTfG4', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d91a7-8263-7200-8565-86014a957c1c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 335, 'total_tokens': 372, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 320}}),
 AIMessage(content='권장됨', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 332, 'pr

In [25]:
examples =[

    {'question': '당신은 어떤 모델입니까?', 'answer': '언어모델'},
    {'question': '인공지능 배워야 하나요?', 'answer': '권장됨'},
    {'question': '파이썬은 뭐가 좋나요?', 'answer': '문법간결'},
]

example_prompt = PromptTemplate(template='질문: {question}\n답변: {answer}')

fewshotprompt = FewShotPromptTemplate(
    examples = examples,
    example_prompt = example_prompt,

    prefix ='주어진 예시 형식을 참고하여 답변하세요.',
    suffix = '질문: {question} \n답변:'
)

test = fewshotprompt.format(question = '원래 말투가 그래?')
print(test)

print(llm.invoke(test).content)

주어진 예시 형식을 참고하여 답변하세요.

질문: 당신은 어떤 모델입니까?
답변: 언어모델

질문: 인공지능 배워야 하나요?
답변: 권장됨

질문: 파이썬은 뭐가 좋나요?
답변: 문법간결

질문: 원래 말투가 그래? 
답변:
상황따라


Chat Message에 Few Shot을 적용하는 경우, 아래와 같이 나타납니다.

In [26]:
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate

example_prompt = ChatPromptTemplate(
        [
        ("human", "{question}"),
        ("ai", "{answer}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(

    example_prompt=example_prompt,
    examples=examples,
)

few_shot_prompt.format_messages()


[HumanMessage(content='당신은 어떤 모델입니까?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='언어모델', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='인공지능 배워야 하나요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='권장됨', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='파이썬은 뭐가 좋나요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='문법간결', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [28]:
final_prompt = ChatPromptTemplate(
    [
        ("system", "다음 예시를 참고하여 같은 형식으로 답변하세요."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)
final_prompt.format_messages(input="휴가 신청은 어떻게 해야 하나요?")
response = llm.invoke(final_prompt.format_messages(input="휴가 신청은 어떻게 해야 하나요?"))
print(response.content)

상사승인후인사팀제출


In [29]:
chain = final_prompt|llm
response = chain.invoke({'input' : "휴가 신청은 어떻게 해야 하나요?"})
print(response.content)

상사승인후인사신청


### 멀티모달 프롬프트 전달하기

멀티모달 모델의 입력은 OpenAI API와 동일합니다.

In [40]:
import base64
import httpx

# Test 이미지 URL
image_url = '''https://i.pinimg.com/originals/d9/8f/ea/d98fea1d8e5d2ef7f844aa8c0a501dd7.jpg'''
response = httpx.get(image_url)

# dog.jpg에 저장
with open('dog1.jpg', 'wb') as file:
    file.write(response.content)

In [31]:
prompt = ChatPromptTemplate(
    [
        ('human', [
            {'type':'image', 'image_url':'{image_url}'},
            {'type':'text', 'text':'이 그림을 자세히 묘사하세요'}
                   
                   ])
    ]
)
llm.invoke(prompt.format_messages(image_url = image_url))



AIMessage(content='사진에는 작고 단단한 체구의 개 한 마리가 풀밭 위에 서 있는 모습이 담겨 있습니다. 전체적으로 연한 황갈색(베이지·크림)에 가까운 털빛을 가지고 있고, 얼굴과 귀 주변은 짙은 갈색 또는 검은색으로 대비되어 있어 전형적인 퍼그(pug) 품종의 외형을 보입니다. 얼굴은 납작하고 주름이 잡혀 있으며, 큰 둥근 눈과 짧은 주둥이, 검은 코가 도드라져 표정이 강하게 보입니다. 귀는 접혀 있고 머리를 약간 돌려 카메라 쪽을 바라보는 듯해 호기심이나 경계심이 느껴집니다.\n\n몸집은 짧고 통통하며 등과 옆구리 라인이 매끄럽고, 짧은 다리가 땅을 단단히 딛고 있는 자세입니다. 꼬리는 등 위로 말려 올라가 있어 퍼그 특유의 말린 꼬리 모양을 보여 줍니다. 털은 짧고 빛을 받아 약간 광택이 나며, 목 주변이나 가슴 부분의 근육 윤곽이 보입니다.\n\n배경은 잔디가 깔린 야외 공간으로 녹색의 풀밭이 넓게 펼쳐져 있고, 사진 전체에 햇빛이 고르게 비쳐 밝고 선명한 분위기입니다. 전경의 개에 초점이 맞춰져 있어 배경은 약간 덜 선명해 보이며, 사진의 구도는 개의 옆모습과 정면이 섞인 3/4 측면으로 안정감 있게 잡혀 있습니다. 전체적으로 건강해 보이고 경계심 있으면서도 호기심 많은 표정의 작은 개를 촬영한 사진입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 794, 'prompt_tokens': 122, 'total_tokens': 916, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_

In [42]:
# 로컬 이미지 전달
with open('./dog1.jpg', 'rb') as image_file:
    image_data = base64.b64encode(image_file.read()).decode('utf-8')


image_prompt = ChatPromptTemplate([
    ('user',[
                {"type": "image_url",
                    "image_url": {"url": "data:image/jpeg;base64,{image_data}"}
                },
                {"type": "text", "text": "{question}"},
             ]
     )])

X = llm.invoke(image_prompt.format_messages(
    question='이 사진을 자세히 묘사해주세요. 강아지의 품종은 설명하지 마세요.',
    image_data=image_data))

print(X.content)

사진에는 작고 동그란 흰 강아지가 중앙에 놓여 있습니다. 전체적으로 털이 아주 풍성하고 부드러워 보이며, 마치 작은 솜뭉치나 구름처럼 둥글게 부풀어 있습니다. 동그란 큰 눈은 검고 반사광이 비쳐 반짝이며, 매우 또렷하고 호기심 어린 표정을 줍니다. 코와 입은 아주 작고 검은 점처럼 보이며 얼굴은 전체적으로 귀엽고 앙증맞은 인상을 줍니다. 네 다리는 짧고 발은 털에 거의 가려져 있어 자세가 아담해 보입니다. 강아지는 벤치의 나무 널빤지 위에 서 있는데, 벤치의 나무색은 약간 바랜 갈색이고 옆에는 검은색의 금속 팔걸이가 보입니다. 배경은 약간 흐릿해 강아지에 시선이 집중되고, 자연광 아래에서 털의 결과 그림자가 은은하게 드러납니다. 전체적으로 포근하고 사랑스러운 순간을 포착한 사진입니다.
